In [34]:
import yfinance as yf
import pandas as pd 

tickers = ["AAPL", "MSFT", "JPM", "NVDA", "SPY", "GLD", "TLT"]

market_data = yf.download(
    tickers,
    start="2020-01-01",
    end="2026-10-01",
    auto_adjust=False
)

market_data.head()


[*********************100%***********************]  7 of 7 completed


Price       Adj Close                                                \
Ticker           AAPL         GLD         JPM        MSFT      NVDA   
Date                                                                  
2020-01-02  72.271561  143.949997  117.899216  151.544250  5.957138   
2020-01-03  71.568916  145.860001  116.343361  149.657303  5.861787   
2020-01-06  72.139175  147.389999  116.250832  150.044128  5.886371   
2020-01-07  71.799942  147.970001  114.274506  148.676056  5.957634   
2020-01-08  72.954910  146.860001  115.165977  151.044205  5.968809   

Price                                   Close                          ...  \
Ticker             SPY         TLT       AAPL         GLD         JPM  ...   
Date                                                                   ...   
2020-01-02  295.391754  111.552567  75.087502  143.949997  141.089996  ...   
2020-01-03  293.154968  113.270523  74.357498  145.860001  138.339996  ...   
2020-01-06  294.273407  112.627312  74.949997  147.389999  138.229996  ...   
2020-01-07  293.445923  112.073624  74.597504  147.970001  135.880005  ...   
2020-01-08  295.009888  111.332771  75.797501  146.860001  136.940002  ...   

Price          Open                             Volume                      \
Ticker         NVDA         SPY         TLT       AAPL       GLD       JPM   
Date                                                                         
2020-01-02  5.96875  323.540009  136.789993  135480400   7733800  10803700   
2020-01-03  5.87750  321.160004  138.270004  146322800  12272800  10386800   
2020-01-06  5.80800  320.489990  139.690002  118387200  14403300  10259000   
2020-01-07  5.95500  323.019989  138.220001  108872000   7978500  10531300   
2020-01-08  5.99400  322.940002  137.929993  132079200  22248500   9695300   

Price                                                
Ticker          MSFT       NVDA       SPY       TLT  
Date                                                 
2020-01-02  22622100  237536000  59151200  11034100  
2020-01-03  21116200  205384000  77709700  12366000  
2020-01-06  20813700  262636000  55653900  11369800  
2020-01-07  21634100  314856000  40496400   8381200  
2020-01-08  27746500  277108000  68296000  10621700  

[5 rows x 42 columns]

In [35]:
print("Shape:", market_data.shape)

print("\nDate range:")
print(market_data.index.min(), "to", market_data.index.max())

print("\nMissing values:")
print(market_data.isna().sum().sum())

print("\nDuplicate dates:")
print(market_data.index.duplicated().sum())

Shape: (1695, 42)

Date range:
2020-01-02 00:00:00 to 2026-09-30 00:00:00

Missing values:
0

Duplicate dates:
0


In [36]:
# Save raw market data
raw_file = "../data/raw/market_data_raw.csv"

market_data.to_csv(raw_file)

print(f"Raw market data saved to: {raw_file}")

Raw market data saved to: ../data/raw/market_data_raw.csv


In [37]:
price_columns = ["Open", "High", "Low", "Close", "Adj Close"]

for price in price_columns:
    invalid_count = (market_data[price] <= 0).sum().sum()
    print(f"{price}: {invalid_count} invalid values")

Open: 0 invalid values
High: 0 invalid values
Low: 0 invalid values
Close: 0 invalid values
Adj Close: 0 invalid values


In [38]:
invalid_high_low = (market_data["High"] < market_data["Low"]).sum().sum()

invalid_high_open = (market_data["High"] < market_data["Open"]).sum().sum()

invalid_high_close = (market_data["High"] < market_data["Close"]).sum().sum()

invalid_low_open = (market_data["Low"] > market_data["Open"]).sum().sum()

invalid_low_close = (market_data["Low"] > market_data["Close"]).sum().sum()

print("High < Low:", invalid_high_low)
print("High < Open:", invalid_high_open)
print("High < Close:", invalid_high_close)
print("Low > Open:", invalid_low_open)
print("Low > Close:", invalid_low_close)

High < Low: 0
High < Open: 0
High < Close: 0
Low > Open: 0
Low > Close: 0


In [39]:
clean_data = (
    market_data
    .stack(level=1, future_stack=True)
    .reset_index()
)

clean_data.columns = [
    "Date",
    "Ticker",
    "Adj_Close",
    "Close",
    "High",
    "Low",
    "Open",
    "Volume"
]

clean_data.head(10)

,Date,Ticker,Adj_Close,Close,High,Low,Open,Volume
0,2020-01-02,AAPL,72.271561,75.087502,75.150002,73.797501,74.059998,135480400
1,2020-01-02,GLD,143.949997,143.949997,144.210007,143.399994,143.860001,7733800
2,2020-01-02,JPM,117.899216,141.089996,141.100006,139.259995,139.789993,10803700
3,2020-01-02,MSFT,151.544250,160.619995,160.729996,158.330002,158.779999,22622100
4,2020-01-02,NVDA,5.957138,5.997750,5.997750,5.918000,5.968750,237536000
5,2020-01-02,SPY,295.391754,324.869995,324.890015,322.529999,323.540009,59151200
6,2020-01-02,TLT,111.552567,137.009995,137.529999,136.669998,136.789993,11034100
7,2020-01-03,AAPL,71.568916,74.357498,75.144997,74.125000,74.287498,146322800
8,2020-01-03,GLD,145.860001,145.860001,146.320007,145.399994,145.750000,12272800
9,2020-01-03,JPM,116.343361,138.339996,139.229996,137.080002,137.500000,10386800


In [40]:
print("Shape:", clean_data.shape)

print("\nColumns:")
print(clean_data.columns.tolist())

print("\nUnique tickers:")
print(clean_data["Ticker"].unique())

print("\nMissing values:")
print(clean_data.isna().sum())

print("\nDuplicate Date-Ticker combinations:")
print(clean_data.duplicated(subset=["Date", "Ticker"]).sum())

Shape: (11865, 8)

Columns:
['Date', 'Ticker', 'Adj_Close', 'Close', 'High', 'Low', 'Open', 'Volume']

Unique tickers:
<StringArray>
['AAPL', 'GLD', 'JPM', 'MSFT', 'NVDA', 'SPY', 'TLT']
Length: 7, dtype: str

Missing values:
Date         0
Ticker       0
Adj_Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

Duplicate Date-Ticker combinations:
0


In [41]:
processed_file = "../data/processed/market_prices_clean.csv"

clean_data.to_csv(processed_file, index=False)

print(f"Processed dataset saved to: {processed_file}")

Processed dataset saved to: ../data/processed/market_prices_clean.csv


In [49]:
clean_data["Daily_Return"] = (clean_data.groupby("Ticker")["Adj_Close"].pct_change())
clean_data.head(10)

,Date,Ticker,Adj_Close,Close,High,Low,Open,Volume,Daily_Return
0,2020-01-02,AAPL,72.271561,75.087502,75.150002,73.797501,74.059998,135480400,NaN
1,2020-01-02,GLD,143.949997,143.949997,144.210007,143.399994,143.860001,7733800,NaN
2,2020-01-02,JPM,117.899216,141.089996,141.100006,139.259995,139.789993,10803700,NaN
3,2020-01-02,MSFT,151.544250,160.619995,160.729996,158.330002,158.779999,22622100,NaN
4,2020-01-02,NVDA,5.957138,5.997750,5.997750,5.918000,5.968750,237536000,NaN
5,2020-01-02,SPY,295.391754,324.869995,324.890015,322.529999,323.540009,59151200,NaN
6,2020-01-02,TLT,111.552567,137.009995,137.529999,136.669998,136.789993,11034100,NaN
7,2020-01-03,AAPL,71.568916,74.357498,75.144997,74.125000,74.287498,146322800,-0.009722
8,2020-01-03,GLD,145.860001,145.860001,146.320007,145.399994,145.750000,12272800,0.013269
9,2020-01-03,JPM,116.343361,138.339996,139.229996,137.080002,137.500000,10386800,-0.013196


In [51]:
print("Missing returns:", clean_data["Daily_Return"].isna().sum())

print("\nReturn statistics:")
print(clean_data["Daily_Return"].describe())

print("\nLargest daily returns:")
print(
    clean_data
    .sort_values("Daily_Return", ascending=False)
    [["Date", "Ticker", "Daily_Return"]]
    .head(10)
)

print("\nLargest daily losses:")
print(
    clean_data
    .sort_values("Daily_Return")
    [["Date", "Ticker", "Daily_Return"]]
    .head(10)
)

Missing returns: 7

Return statistics:
count    11858.000000
mean         0.000942
std          0.019247
min         -0.184521
25%         -0.007353
50%          0.000917
75%          0.009136
max          0.243696
Name: Daily_Return, dtype: float64

Largest daily returns:
            Date Ticker  Daily_Return
5989  2023-05-25   NVDA      0.243696
9272  2025-04-09   NVDA      0.187227
345   2020-03-13    JPM      0.180125
396   2020-03-24   NVDA      0.171564
7291  2024-02-22   NVDA      0.164009
11560 2026-07-30   MSFT      0.155067
9268  2025-04-09   AAPL      0.153288
5051  2022-11-10   NVDA      0.143293
346   2020-03-13   MSFT      0.142169
5541  2023-02-23   NVDA      0.140214

Largest daily losses:
            Date Ticker  Daily_Return
354   2020-03-16   NVDA     -0.184521
8915  2025-01-27   NVDA     -0.169682
352   2020-03-16    JPM     -0.149649
353   2020-03-16   MSFT     -0.147390
317   2020-03-09    JPM     -0.135455
350   2020-03-16   AAPL     -0.128647
340   2020-03-12   

In [53]:
returns_file = "../data/processed/market_returns.csv"

clean_data.to_csv(returns_file, index=False)

print(f"Returns dataset saved to: {returns_file}")

Returns dataset saved to: ../data/processed/market_returns.csv


In [54]:
annual_metrics = (
    clean_data
    .groupby("Ticker")["Daily_Return"]
    .agg(
        Average_Daily_Return="mean",
        Daily_Volatility="std"
    )
)

annual_metrics["Annualized_Return"] = (
    annual_metrics["Average_Daily_Return"] * 252
)

annual_metrics["Annualized_Volatility"] = (
    annual_metrics["Daily_Volatility"] * (252 ** 0.5)
)

annual_metrics = annual_metrics.round(4)

annual_metrics

,Average_Daily_Return,Daily_Volatility,Annualized_Return,Annualized_Volatility
Ticker,,,,
AAPL,0.0011,0.0197,0.2762,0.3128
GLD,0.0006,0.0117,0.1621,0.1855
JPM,0.0008,0.0192,0.2000,0.3053
MSFT,0.0009,0.0191,0.2274,0.3036
NVDA,0.0027,0.0326,0.6759,0.5174
SPY,0.0006,0.0126,0.1613,0.2005
TLT,-0.0002,0.0104,-0.0407,0.1646


In [56]:
portfolio_weights = pd.DataFrame({
    "Ticker": ["AAPL", "MSFT", "JPM", "NVDA", "SPY", "GLD", "TLT"],
    "Weight": [0.15, 0.15, 0.15, 0.15, 0.20, 0.10, 0.10]
})

print("Total portfolio weight:", portfolio_weights["Weight"].sum())

portfolio_weights

Total portfolio weight: 1.0


,Ticker,Weight
0,AAPL,0.15
1,MSFT,0.15
2,JPM,0.15
3,NVDA,0.15
4,SPY,0.20
5,GLD,0.10
6,TLT,0.10


In [57]:
asset_classes = {
    "AAPL": "Equity",
    "MSFT": "Equity",
    "JPM": "Equity",
    "NVDA": "Equity",
    "SPY": "Equity ETF",
    "GLD": "Commodity ETF",
    "TLT": "Bond ETF"
}

portfolio_weights["Asset_Class"] = (
    portfolio_weights["Ticker"].map(asset_classes)
)

portfolio_weights

,Ticker,Weight,Asset_Class
0,AAPL,0.15,Equity
1,MSFT,0.15,Equity
2,JPM,0.15,Equity
3,NVDA,0.15,Equity
4,SPY,0.20,Equity ETF
5,GLD,0.10,Commodity ETF
6,TLT,0.10,Bond ETF


In [58]:
portfolio_file = "../data/processed/portfolio_holdings.csv"

portfolio_weights.to_csv(portfolio_file, index=False)

print(f"Portfolio holdings saved to: {portfolio_file}")

Portfolio holdings saved to: ../data/processed/portfolio_holdings.csv


In [60]:
portfolio_data = clean_data.merge(portfolio_weights[["Ticker", "Weight"]],on="Ticker",how="left")

portfolio_data.head()

,Date,Ticker,Adj_Close,Close,High,Low,Open,Volume,Daily_Return,Weight
0,2020-01-02,AAPL,72.271561,75.087502,75.150002,73.797501,74.059998,135480400,NaN,0.15
1,2020-01-02,GLD,143.949997,143.949997,144.210007,143.399994,143.860001,7733800,NaN,0.10
2,2020-01-02,JPM,117.899216,141.089996,141.100006,139.259995,139.789993,10803700,NaN,0.15
3,2020-01-02,MSFT,151.544250,160.619995,160.729996,158.330002,158.779999,22622100,NaN,0.15
4,2020-01-02,NVDA,5.957138,5.997750,5.997750,5.918000,5.968750,237536000,NaN,0.15


In [62]:
portfolio_data["Weighted_Return"] = (portfolio_data["Daily_Return"] *portfolio_data["Weight"])

portfolio_data.head(10)

,Date,Ticker,Adj_Close,Close,High,Low,Open,Volume,Daily_Return,Weight,Weighted_Return
0,2020-01-02,AAPL,72.271561,75.087502,75.150002,73.797501,74.059998,135480400,NaN,0.15,NaN
1,2020-01-02,GLD,143.949997,143.949997,144.210007,143.399994,143.860001,7733800,NaN,0.10,NaN
2,2020-01-02,JPM,117.899216,141.089996,141.100006,139.259995,139.789993,10803700,NaN,0.15,NaN
3,2020-01-02,MSFT,151.544250,160.619995,160.729996,158.330002,158.779999,22622100,NaN,0.15,NaN
4,2020-01-02,NVDA,5.957138,5.997750,5.997750,5.918000,5.968750,237536000,NaN,0.15,NaN
5,2020-01-02,SPY,295.391754,324.869995,324.890015,322.529999,323.540009,59151200,NaN,0.20,NaN
6,2020-01-02,TLT,111.552567,137.009995,137.529999,136.669998,136.789993,11034100,NaN,0.10,NaN
7,2020-01-03,AAPL,71.568916,74.357498,75.144997,74.125000,74.287498,146322800,-0.009722,0.15,-0.001458
8,2020-01-03,GLD,145.860001,145.860001,146.320007,145.399994,145.750000,12272800,0.013269,0.10,0.001327
9,2020-01-03,JPM,116.343361,138.339996,139.229996,137.080002,137.500000,10386800,-0.013196,0.15,-0.001979


In [64]:
portfolio_returns = (portfolio_data.groupby("Date")["Weighted_Return"].sum().reset_index())

portfolio_returns.rename(columns={"Weighted_Return": "Portfolio_Return"},inplace=True)

portfolio_returns.head(10)

,Date,Portfolio_Return
0,2020-01-02,0.000000
1,2020-01-03,-0.006354
2,2020-01-06,0.003337
3,2020-01-07,-0.003468
4,2020-01-08,0.005908
5,2020-01-09,0.008397
6,2020-01-10,-0.000132
7,2020-01-13,0.011287
8,2020-01-14,-0.004010
9,2020-01-15,-0.001258


In [65]:
print("Portfolio observations:", len(portfolio_returns))

print("\nMissing portfolio returns:")
print(portfolio_returns["Portfolio_Return"].isna().sum())

print("\nPortfolio return statistics:")
print(portfolio_returns["Portfolio_Return"].describe())

Portfolio observations: 1695

Missing portfolio returns:
0

Portfolio return statistics:
count    1695.000000
mean        0.000997
std         0.013237
min        -0.108084
25%        -0.005359
50%         0.001241
75%         0.007894
max         0.103653
Name: Portfolio_Return, dtype: float64


In [67]:
portfolio_returns = (portfolio_data.pivot(index="Date", columns="Ticker", values="Daily_Return"))

portfolio_returns = portfolio_returns.dropna()

weights = portfolio_weights.set_index("Ticker")["Weight"]

portfolio_returns["Portfolio_Return"] = (portfolio_returns[weights.index] * weights).sum(axis=1)

portfolio_returns = (portfolio_returns[["Portfolio_Return"]].reset_index())

print("Valid portfolio return observations:", len(portfolio_returns))
print("Missing portfolio returns:", portfolio_returns["Portfolio_Return"].isna().sum())

portfolio_returns.head()

Valid portfolio return observations: 1694
Missing portfolio returns: 0


Ticker,Date,Portfolio_Return
0,2020-01-03,-0.006354
1,2020-01-06,0.003337
2,2020-01-07,-0.003468
3,2020-01-08,0.005908
4,2020-01-09,0.008397


In [68]:
daily_returns = portfolio_returns["Portfolio_Return"]

annualized_return = daily_returns.mean() * 252
annualized_volatility = daily_returns.std() * (252 ** 0.5)

print(f"Annualized Return: {annualized_return:.2%}")
print(f"Annualized Volatility: {annualized_volatility:.2%}")

Annualized Return: 25.13%
Annualized Volatility: 21.02%


In [70]:
risk_free_rate = 0

sharpe_ratio = (annualized_return - risk_free_rate) / annualized_volatility

print(f"Sharpe Ratio: {sharpe_ratio:.2f}")

Sharpe Ratio: 1.20


In [72]:
portfolio_returns["Cumulative_Return"] = ((1 + portfolio_returns["Portfolio_Return"]).cumprod())
portfolio_returns.head()

Ticker,Date,Portfolio_Return,Cumulative_Return
0,2020-01-03,-0.006354,0.993646
1,2020-01-06,0.003337,0.996962
2,2020-01-07,-0.003468,0.993505
3,2020-01-08,0.005908,0.999375
4,2020-01-09,0.008397,1.007766


In [74]:
portfolio_returns["Running_Peak"] = (portfolio_returns["Cumulative_Return"].cummax())
portfolio_returns.head()

Ticker,Date,Portfolio_Return,Cumulative_Return,Running_Peak
0,2020-01-03,-0.006354,0.993646,0.993646
1,2020-01-06,0.003337,0.996962,0.996962
2,2020-01-07,-0.003468,0.993505,0.996962
3,2020-01-08,0.005908,0.999375,0.999375
4,2020-01-09,0.008397,1.007766,1.007766


In [76]:
portfolio_returns["Drawdown"] = (portfolio_returns["Cumulative_Return"]/ portfolio_returns["Running_Peak"]) - 1
portfolio_returns.head()

Ticker,Date,Portfolio_Return,Cumulative_Return,Running_Peak,Drawdown
0,2020-01-03,-0.006354,0.993646,0.993646,0.000000
1,2020-01-06,0.003337,0.996962,0.996962,0.000000
2,2020-01-07,-0.003468,0.993505,0.996962,-0.003468
3,2020-01-08,0.005908,0.999375,0.999375,0.000000
4,2020-01-09,0.008397,1.007766,1.007766,0.000000


In [78]:
maximum_drawdown = portfolio_returns["Drawdown"].min()

max_drawdown_date = portfolio_returns.loc[portfolio_returns["Drawdown"].idxmin(),"Date"]

print(f"Maximum Drawdown: {maximum_drawdown:.2%}")
print(f"Maximum Drawdown Date: {max_drawdown_date.date()}")

Maximum Drawdown: -32.18%
Maximum Drawdown Date: 2022-10-11


In [80]:
portfolio_returns["Rolling_30D_Volatility"] = (portfolio_returns["Portfolio_Return"].rolling(window=30).std() * (252 ** 0.5))

portfolio_returns[["Date","Portfolio_Return","Rolling_30D_Volatility"]].tail(10)

Ticker,Date,Portfolio_Return,Rolling_30D_Volatility
1684,2026-09-17,0.013403,0.110370
1685,2026-09-18,0.000889,0.110253
1686,2026-09-21,0.011198,0.112430
1687,2026-09-22,-0.004554,0.112734
1688,2026-09-23,-0.008533,0.115494
1689,2026-09-24,-0.003207,0.115718
1690,2026-09-25,0.011516,0.120064
1691,2026-09-28,-0.009808,0.123610
1692,2026-09-29,-0.005411,0.123199
1693,2026-09-30,0.000202,0.122921


In [82]:
highest_volatility = (portfolio_returns.dropna(subset=["Rolling_30D_Volatility"]).nlargest(10, "Rolling_30D_Volatility")[["Date", "Rolling_30D_Volatility"]])

highest_volatility

Ticker,Date,Rolling_30D_Volatility
64,2020-04-06,0.773368
66,2020-04-08,0.772145
65,2020-04-07,0.769756
67,2020-04-09,0.763484
68,2020-04-13,0.763404
69,2020-04-14,0.756692
70,2020-04-15,0.754358
63,2020-04-03,0.751414
62,2020-04-02,0.751385
71,2020-04-16,0.748211


In [84]:
portfolio_risk_file = "../data/processed/portfolio_returns_risk.csv"

portfolio_returns.to_csv(portfolio_risk_file,index=False)

print(f"Portfolio risk dataset saved to: {portfolio_risk_file}")

Portfolio risk dataset saved to: ../data/processed/portfolio_returns_risk.csv


In [86]:
var_95 = daily_returns.quantile(0.05)

print(f"Historical 1-Day VaR (95%): {var_95:.2%}")

Historical 1-Day VaR (95%): -1.87%


In [88]:
var_95 = daily_returns.quantile(0.05)

expected_shortfall_95 = daily_returns[daily_returns <= var_95].mean()

print(f"Historical 1-Day VaR (95%): {var_95:.2%}")
print(f"Expected Shortfall (95%): {expected_shortfall_95:.2%}")

Historical 1-Day VaR (95%): -1.87%
Expected Shortfall (95%): -2.98%


In [90]:
asset_returns = (clean_data.pivot(index="Date", columns="Ticker", values="Daily_Return").dropna())

asset_returns.head()

Ticker,AAPL,GLD,JPM,MSFT,NVDA,SPY,TLT
Date,,,,,,,
2020-01-03,-0.009722,0.013269,-0.013196,-0.012451,-0.016006,-0.007572,0.015400
2020-01-06,0.007968,0.010490,-0.000795,0.002585,0.004194,0.003815,-0.005679
2020-01-07,-0.004702,0.003935,-0.017001,-0.009118,0.012106,-0.002812,-0.004916
2020-01-08,0.016086,-0.007502,0.007801,0.015928,0.001876,0.005330,-0.006610
2020-01-09,0.021241,-0.005652,0.003651,0.012493,0.010982,0.006781,0.003510


In [91]:
covariance_matrix = asset_returns.cov()

covariance_matrix

Ticker,AAPL,GLD,JPM,MSFT,NVDA,SPY,TLT
Ticker,,,,,,,
AAPL,0.000388,0.000021,0.000153,0.000241,0.000348,0.000188,-0.000011
GLD,0.000021,0.000137,0.000004,0.000021,0.000047,0.000023,0.000025
JPM,0.000153,0.000004,0.000370,0.000143,0.000207,0.000169,-0.000048
MSFT,0.000241,0.000021,0.000143,0.000366,0.000388,0.000182,-0.000015
NVDA,0.000348,0.000047,0.000207,0.000388,0.001062,0.000288,-0.000008
SPY,0.000188,0.000023,0.000169,0.000182,0.000288,0.000160,-0.000013
TLT,-0.000011,0.000025,-0.000048,-0.000015,-0.000008,-0.000013,0.000108


In [93]:
weights = portfolio_weights.set_index("Ticker")["Weight"]

portfolio_variance = (weights.T @ covariance_matrix.loc[weights.index, weights.index]@ weights)

portfolio_daily_volatility = portfolio_variance ** 0.5

print(f"Portfolio Daily Volatility: {portfolio_daily_volatility:.4%}")

Portfolio Daily Volatility: 1.3241%


In [96]:
portfolio_covariance = covariance_matrix.loc[weights.index,weights.index]

portfolio_volatility = (weights.T @ portfolio_covariance @ weights) ** 0.5

marginal_contribution = (portfolio_covariance @ weights) / portfolio_volatility

risk_contribution = (weights * marginal_contribution)

risk_contribution_pct = (risk_contribution / portfolio_volatility)

risk_contribution_df = pd.DataFrame({"Ticker": weights.index,"Weight": weights.values,"Risk_Contribution": risk_contribution.values,"Risk_Contribution_Pct": risk_contribution_pct.values})

risk_contribution_df

,Ticker,Weight,Risk_Contribution,Risk_Contribution_Pct
0,AAPL,0.15,0.002358,0.178053
1,MSFT,0.15,0.002350,0.177480
2,JPM,0.15,0.001816,0.137112
3,NVDA,0.15,0.004103,0.309885
4,SPY,0.20,0.002366,0.178695
5,GLD,0.10,0.000261,0.019747
6,TLT,0.10,-0.000013,-0.000973


In [98]:
print("Total Risk Contribution:",risk_contribution_df["Risk_Contribution_Pct"].sum())

Total Risk Contribution: 0.9999999999999999


In [106]:
risk_contribution_df = (risk_contribution_df.sort_values("Risk_Contribution_Pct", ascending=False).reset_index(drop=True))

risk_contribution_df["Risk_Contribution_Pct"] = (risk_contribution_df["Risk_Contribution_Pct"] * 100)

risk_contribution_df["Weight"] = (risk_contribution_df["Weight"] * 100)

risk_contribution_df

,Ticker,Weight,Risk_Contribution,Risk_Contribution_Pct
0,NVDA,150000.0,0.004103,309885.067493
1,SPY,200000.0,0.002366,178695.349129
2,AAPL,150000.0,0.002358,178052.666839
3,MSFT,150000.0,0.002350,177480.196090
4,JPM,150000.0,0.001816,137112.110911
5,GLD,100000.0,0.000261,19747.426529
6,TLT,100000.0,-0.000013,-972.816991


In [107]:
print("Risk contribution total:",risk_contribution_df["Risk_Contribution_Pct"].sum())

Risk contribution total: 1000000.0


In [112]:
portfolio_covariance = covariance_matrix.loc[weights.index,weights.index]

portfolio_volatility = (weights.T @ portfolio_covariance @ weights) ** 0.5

marginal_contribution = (portfolio_covariance @ weights) / portfolio_volatility

risk_contribution = (weights * marginal_contribution)

risk_contribution_pct = (risk_contribution / portfolio_volatility)

risk_contribution_df = pd.DataFrame({"Ticker": weights.index,"Weight": weights.values,"Risk_Contribution": risk_contribution.values,"Risk_Contribution_Pct": risk_contribution_pct.values})

print("Risk contribution total:",risk_contribution_df["Risk_Contribution_Pct"].sum())

Risk contribution total: 0.9999999999999999


In [113]:
risk_contribution_display = (risk_contribution_df.sort_values("Risk_Contribution_Pct", ascending=False).reset_index(drop=True))

risk_contribution_display["Weight"] = (risk_contribution_display["Weight"] * 100)

risk_contribution_display["Risk_Contribution_Pct"] = (risk_contribution_display["Risk_Contribution_Pct"] * 100)

risk_contribution_display

,Ticker,Weight,Risk_Contribution,Risk_Contribution_Pct
0,NVDA,15.0,0.004103,30.988507
1,SPY,20.0,0.002366,17.869535
2,AAPL,15.0,0.002358,17.805267
3,MSFT,15.0,0.002350,17.748020
4,JPM,15.0,0.001816,13.711211
5,GLD,10.0,0.000261,1.974743
6,TLT,10.0,-0.000013,-0.097282


In [115]:
print("Risk contribution total (%):",risk_contribution_display["Risk_Contribution_Pct"].sum())

Risk contribution total (%): 100.0


In [116]:
stress_scenarios = {
    "Market Crash": {
        "AAPL": -0.10,
        "MSFT": -0.10,
        "JPM": -0.10,
        "NVDA": -0.10,
        "SPY": -0.10,
        "GLD": 0.00,
        "TLT": 0.00
    },
    
    "Severe Market Crash": {
        "AAPL": -0.20,
        "MSFT": -0.20,
        "JPM": -0.20,
        "NVDA": -0.20,
        "SPY": -0.20,
        "GLD": 0.00,
        "TLT": 0.00
    },
    
    "Commodity Shock": {
        "AAPL": 0.00,
        "MSFT": 0.00,
        "JPM": 0.00,
        "NVDA": 0.00,
        "SPY": 0.00,
        "GLD": -0.15,
        "TLT": 0.00
    }
}

print("Stress scenarios created:", len(stress_scenarios))

Stress scenarios created: 3


In [121]:
stress_results = []

for scenario, shocks in stress_scenarios.items():
    portfolio_impact = 0
    
    for ticker, shock in shocks.items():
        weight = weights[ticker]
        portfolio_impact += weight * shock
    
    stress_results.append({
        "Scenario": scenario,
        "Portfolio_Impact": portfolio_impact
    })

stress_test_results = pd.DataFrame(stress_results)

stress_test_results

,Scenario,Portfolio_Impact
0,Market Crash,-0.080
1,Severe Market Crash,-0.160
2,Commodity Shock,-0.015


In [123]:
stress_test_results["Portfolio_Impact_Pct"] = (
    stress_test_results["Portfolio_Impact"] * 100)

stress_test_results

,Scenario,Portfolio_Impact,Portfolio_Impact_Pct
0,Market Crash,-0.080,-8.0
1,Severe Market Crash,-0.160,-16.0
2,Commodity Shock,-0.015,-1.5


In [124]:
portfolio_summary = pd.DataFrame({
    "Metric": [
        "Annualized Return",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown",
        "Historical VaR 95%",
        "Expected Shortfall 95%"
    ],
    "Value": [
        annualized_return,
        annualized_volatility,
        sharpe_ratio,
        maximum_drawdown,
        var_95,
        expected_shortfall_95
    ]
})

portfolio_summary

,Metric,Value
0,Annualized Return,0.251301
1,Annualized Volatility,0.210199
2,Sharpe Ratio,1.195542
3,Maximum Drawdown,-0.321823
4,Historical VaR 95%,-0.018691
5,Expected Shortfall 95%,-0.029821


In [125]:
summary_file = "../data/processed/portfolio_risk_summary.csv"

portfolio_summary.to_csv(
    summary_file,
    index=False
)

print(f"Portfolio risk summary saved to: {summary_file}")

Portfolio risk summary saved to: ../data/processed/portfolio_risk_summary.csv
